# PlasmaCAD — Nesting con algoritmo genético

Pasos:
1. **Entorno → Cambiar tipo de entorno → GPU T4** (opcional, pero el nesting corre ~3-5× más rápido).
2. Subí los DXF del trabajo a `Mi unidad/PlasmaCAD/trabajos/<TRABAJO>/entrada/`.
   La cantidad de cada pieza va en el nombre del archivo: `..._CANT_40.dxf`.
   Etiquetas opcionales en el nombre: `NOROT` (no girar), `ROT90` (solo 0/90/180/270), `ROT180` (solo 0/180).
3. Completá el formulario de la celda 3 y ejecutá **Entorno → Ejecutar todas** (Ctrl+F9).
4. Los DXF por chapa, la imagen y el resumen quedan en `.../<TRABAJO>/salida/` en tu Drive.

In [ ]:
#@title 1 · Preparar entorno (GPU, librerías y código)
import subprocess, sys, os
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'no hay GPU, se usa CPU')
!pip -q install ezdxf shapely
if gpu:
    try:
        import cupy  # viene instalado en los entornos GPU de Colab
    except ImportError:
        !pip -q install cupy-cuda12x

# --- Código desde GitHub. Repo público: no hace falta token.
# Repo privado: token guardado en Secretos de Colab (icono de llave) con el nombre GITHUB_TOKEN
REPO = 'github.com/Franrnn/plasma-nesting.git'   #@param {type:"string"}
RAMA = 'main'                                    #@param {type:"string"}
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://x-access-token:{token}@{REPO}' if token else f'https://{REPO}'
if os.path.isdir('/content/plasma-nesting'):
    !git -C /content/plasma-nesting pull -q
else:
    !git clone -q -b {RAMA} {url} /content/plasma-nesting
sys.path.insert(0, '/content/plasma-nesting')
from nesting.motor import GPU_DISPONIBLE
print('Motor listo | GPU para nesting:', GPU_DISPONIBLE)

In [ ]:
#@title 2 · Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/PlasmaCAD/trabajos'
os.makedirs(BASE, exist_ok=True)
print('Trabajos disponibles:', sorted(os.listdir(BASE)))

In [ ]:
#@title 3 · Parámetros del nesting
TRABAJO = 'T-3094_bridas_5-16'      #@param {type:"string"}
LARGO_CHAPA = 3000                  #@param {type:"number"}
ANCHO_CHAPA = 1500                  #@param {type:"number"}
ESPESOR_MM = 7.94                   #@param {type:"number"}
SEPARACION_PIEZAS = 10              #@param {type:"number"}
MARGEN_BORDE = 10                   #@param {type:"number"}
PASO_ANGULO = 15                    #@param [90, 45, 15, 5] {type:"raw"}
PIEZA_EN_PIEZA = True               #@param {type:"boolean"}
MINUTOS_MAX = 10                    #@param {type:"slider", min:2, max:120, step:1}
REINICIOS = 3                       #@param {type:"slider", min:1, max:10, step:1}
GENERACIONES = 300                  #@param {type:"integer"}
POBLACION = 40                      #@param {type:"integer"}
CAPA_UNICA = ''                     #@param {type:"string"}
DIBUJAR_CONTORNO_CHAPA = False      #@param {type:"boolean"}

ENTRADA = f'{BASE}/{TRABAJO}/entrada'
SALIDA = f'{BASE}/{TRABAJO}/salida'
assert os.path.isdir(ENTRADA), f'No existe {ENTRADA}: creala y subí los DXF'
print(len([f for f in os.listdir(ENTRADA) if f.lower().endswith('.dxf')]), 'archivos DXF en', ENTRADA)

In [ ]:
#@title 4 · Ejecutar nesting
from nesting.motor import ConfigNesting
from nesting.pipeline import ejecutar
cfg = ConfigNesting(largo=LARGO_CHAPA, ancho=ANCHO_CHAPA, margen=MARGEN_BORDE, gap=SEPARACION_PIEZAS,
                    paso_angulo=int(PASO_ANGULO), pieza_en_pieza=PIEZA_EN_PIEZA, poblacion=POBLACION,
                    generaciones=GENERACIONES, tiempo_max_s=MINUTOS_MAX * 60, espesor=ESPESOR_MM,
                    procesos=os.cpu_count() or 2, reinicios=REINICIOS)
resumen = ejecutar(ENTRADA, SALIDA, cfg, prefijo=TRABAJO, capa_unica=CAPA_UNICA or None,
                   contorno_chapa=DIBUJAR_CONTORNO_CHAPA)

In [ ]:
#@title 5 · Ver resultado y descargar
from IPython.display import Image, display
import json, shutil
display(Image(f'{SALIDA}/{TRABAJO}_nesting.png', width=1100))
print(json.dumps({k: resumen[k] for k in ('chapas_usadas', 'retazo_mm', 'detalle_chapas', 'separacion_minima_mm', 'verificado')}, indent=1, ensure_ascii=False))
zip_path = shutil.make_archive(f'/content/{TRABAJO}_nesting', 'zip', SALIDA)
from google.colab import files
files.download(zip_path)   # además queda todo guardado en Drive